## 1. Data Loading and Preprocessing
- Load the TripAdvisor CSV file.  
- Rename columns: `likes, rating, review_text, group_type, reviewer_location`.  
- Handle encoding issues (`cp1252`), remove nulls, and check basic stats.  
- Optional: remove non-English reviews or normalize text.

## 2. Exploratory Overview
- Show total number of reviews, unique trip types, and top reviewer locations.  
- Display a few example reviews to confirm structure and encoding.  
- Compute word frequency or length distribution for context.

In [1]:
import pandas as pd
from collections import Counter
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv("../Data/Raw/ushmm_tripadvisor_20250908.csv", header=None, encoding="latin1")

# Quick look
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11367 entries, 0 to 11366
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   0       11367 non-null  int64 
 1   1       11367 non-null  int64 
 2   2       11367 non-null  object
 3   3       10949 non-null  object
 4   4       8791 non-null   object
dtypes: int64(2), object(3)
memory usage: 444.1+ KB
None


In [7]:
print(df.columns.tolist())
print(df.head())
print(df.shape)
print(df.head(3).to_string())

[0, 1, 2, 3, 4]
   0  1                                                  2        3  \
0  0  5  We spent four hours and could have spent eight...  COUPLES   
1  1  5  We are Germans visiting this museum. Most of t...   FAMILY   
2  0  5  This museum is incredible and incredibly impor...   FAMILY   
3  0  5  I visited the Holocaust Museum in August 2024....     SOLO   
4  0  4  I learned so much. The place has a very sad vi...  FRIENDS   

                         4  
0        Bradley, Illinois  
1      Heidelberg, Germany  
2        Otsego, Minnesota  
3  New York City, New York  
4       Lewiston, New York  
(11367, 5)
   0  1                                                                                                                                                                                                                                                                                                                                                                             

In [4]:
# Rename columns
df.columns = ["likes", "rating", "review_text", "group_type", "reviewer_location"]

print(df.head())

   likes  rating                                        review_text  \
0      0       5  We spent four hours and could have spent eight...   
1      1       5  We are Germans visiting this museum. Most of t...   
2      0       5  This museum is incredible and incredibly impor...   
3      0       5  I visited the Holocaust Museum in August 2024....   
4      0       4  I learned so much. The place has a very sad vi...   

  group_type        reviewer_location  
0    COUPLES        Bradley, Illinois  
1     FAMILY      Heidelberg, Germany  
2     FAMILY        Otsego, Minnesota  
3       SOLO  New York City, New York  
4    FRIENDS       Lewiston, New York  


In [5]:
print(df.columns)

# Number of reviews
tot_rev=len(df)
print("Total reviews:", len(df))

Index(['likes', 'rating', 'review_text', 'group_type', 'reviewer_location'], dtype='object')
Total reviews: 11367


In [6]:
# Keeping only english reviews as part of our analysis for the time being
import langdetect

def detect_lang(text):
    try:
        return langdetect.detect(str(text))
    except:
        return "error"

df["lang"] = df["review_text"].dropna().apply(detect_lang)
df = df[df["lang"] == "en"]

In [7]:
df = df.reset_index(drop=True)

In [8]:
print(df.count())

likes                10116
rating               10116
review_text          10116
group_type            9735
reviewer_location     7780
lang                 10116
dtype: int64


In [9]:

print("Number of english reviews:", len(df))
p=((tot_rev-len(df))/tot_rev)*100

print("Number of non-english reviews:",tot_rev-len(df),"Approx '%' removed", p)

Number of english reviews: 10116
Number of non-english reviews: 1251 Approx '%' removed 11.0055423594616


In [10]:
print("Total reviews:", len(df))
print("Average rating:", df["rating"].mean())
print("\nReview counts per rating:", df["rating"].value_counts().sort_index())

Total reviews: 10116
Average rating: 4.714709371293001

Review counts per rating: rating
1      75
2     117
3     419
4    1397
5    8108
Name: count, dtype: int64


In [11]:
print("\nTrip type distribution:\n", df["group_type"].value_counts())
print("\nTop reviewer locations:\n", df["reviewer_location"].value_counts().head(10))


Trip type distribution:
 group_type
FAMILY      3209
COUPLES     2663
FRIENDS     1332
NONE        1303
SOLO         820
BUSINESS     408
Name: count, dtype: int64

Top reviewer locations:
 reviewer_location
Washington DC, District of Columbia    330
New York City, New York                117
London, United Kingdom                  76
Chicago, Illinois                       71
Toronto, Canada                         71
Atlanta, Georgia                        70
Boston, Massachusetts                   62
Orlando, Florida                        49
Philadelphia, Pennsylvania              47
Minneapolis, Minnesota                  47
Name: count, dtype: int64


In [12]:
from collections import Counter

all_text = " ".join(df['review_text'].dropna().astype(str))
words = [w.lower() for w in all_text.split()]
common_words = Counter(words).most_common(20)
print("20 most common words among the reviews:", common_words)

20 most common words among the reviews: [('the', 46621), ('to', 27107), ('and', 25177), ('of', 19453), ('a', 19279), ('is', 13115), ('i', 12124), ('it', 11581), ('in', 11458), ('this', 11216), ('you', 10797), ('was', 9102), ('museum', 8603), ('that', 8371), ('we', 7314), ('for', 7057), ('but', 5582), ('very', 5563), ('not', 4995), ('so', 4781)]


In [13]:
# Numeric sanity
for col in ["likes", "rating"]:
    if col in df.columns:
        print(col, ":\n", pd.to_numeric(df[col], errors="coerce").describe(),"\n")

likes :
 count    10116.000000
mean         0.450474
std          1.008427
min          0.000000
25%          0.000000
50%          0.000000
75%          1.000000
max         34.000000
Name: likes, dtype: float64 

rating :
 count    10116.000000
mean         4.714709
std          0.667193
min          1.000000
25%          5.000000
50%          5.000000
75%          5.000000
max          5.000000
Name: rating, dtype: float64 



In [14]:
# Pull a few compelling snippets that illustrate emotion/moral duty/learning
examples = df["review_text"].dropna().astype(str).sample(3, random_state=42).to_list()
for i, ex in enumerate(examples, 1):
    print(f"\nExample {i}:\n{ex[:600]}")


Example 1:
The US Holocaust Memorial Museum is an absolute must visit!  This museum has been established for many substantial reasons, and exhibits many powerful galleries that recall visitors to find a connection between their own lives and those that were destroyed throughout the Holocaust.

Example 2:
The museum is divided in three floors: the first shows the rise of Hitler and the Nazis to power and their crazy racial ideology; the middle one shows the assault on Jews and mass murder, as well as Nazi invasions; and the last one depicts the liberation of the camps and aftermath. Really instructive! We were only uncomfortable by the number of people inside the museum... we made reservations online and expected the number of visitors would be limited. Plan to spend at least three hours in the museum.

Example 3:
It's done very well. Really glad we went to see it. Do buy your ticket online beforehand to save time. 

Only problem was it was very busy with all the obese Americans blocki

In [15]:
stopwords = set(["the","to","and","a","of","is","in","it","this","that","for","on","we","i","you","be","with","as","at","so","are","was","but","have","there"])
all_text = " ".join(df['review_text'].dropna().astype(str)).lower().split()
words = [w.strip(".,!?\"'()") for w in all_text if w not in stopwords]
print("Next 20 most common words that are actually meaningful for this exercise", Counter(words).most_common(20))

Next 20 most common words that are actually meaningful for this exercise [('museum', 11446), ('very', 5605), ('not', 5140), ('holocaust', 4576), ('time', 4511), ('see', 4286), ('were', 4119), ('visit', 3333), ('they', 3248), ('all', 3218), ('my', 3109), ('about', 3087), ('an', 3085), ('people', 3061), ('through', 3048), ('get', 2843), ('what', 2793), ('one', 2740), ('if', 2663), ('well', 2658)]


In [16]:
df.drop(columns=["lang"], inplace=True)
df.to_csv("../Data/Processed/ushmm_tripadvisor_eng.csv", index=False)